# Fiche 6 — Interroger une table : SELECT, WHERE, GROUP BY, HAVING, ORDER BY

| | |
|---|---|
| 🎯 **Objectif** | Écrire une requête d'agrégation sur une table et savoir où placer chaque filtre |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 5 (une base chargée) |
| 🔗 **Dans le projet** | Partie 3 (requêtes) |

> **L'idée clé.** Une requête se lit dans l'ordre où SQLite l'exécute : FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY, LIMIT.

In [1]:
import sqlite3

import pandas as pd

# Ouvre TA base si elle est prête ; sinon, la base de démonstration du dossier (demande-la à ton mentor
# si elle ne t'a pas été transmise). Ouverture en lecture seule : les exemples ne peuvent rien abîmer.
base = sqlite3.connect("file:../data/processed/dataimmo.sqlite?mode=ro", uri=True)


def sql(requete: str) -> pd.DataFrame:
    return pd.read_sql(requete, base)

sql("SELECT * FROM vente LIMIT 3")

,id_vente,id_bien,date_vente,valeur_fonciere,nombre_lots
0,1,1,2020-01-02,165000.0,2
1,2,2,2020-01-02,355680.0,2
2,3,3,2020-01-02,229500.0,1


## 1. Sélectionner et filtrer : SELECT, WHERE, ORDER BY, LIMIT

In [2]:
sql("""
SELECT id_vente, date_vente, valeur_fonciere, nombre_lots
FROM vente
WHERE nombre_lots >= 5
ORDER BY valeur_fonciere DESC
LIMIT 5
""")

,id_vente,date_vente,valeur_fonciere,nombre_lots
0,3625,2020-01-22,8577713.0,5
1,9988,2020-02-19,7600000.0,5
2,1924,2020-01-14,7050000.0,8
3,12186,2020-02-28,2686000.0,9
4,26259,2020-05-29,2600000.0,5


## 2. Regrouper et agréger : GROUP BY

`GROUP BY` forme des paquets de lignes ; une **fonction d'agrégat** (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) résume
chaque paquet.

In [3]:
sql("""
SELECT
    nombre_lots,
    COUNT(*)                         AS nb_ventes,
    ROUND(AVG(valeur_fonciere), 0)   AS valeur_moyenne
FROM vente
GROUP BY nombre_lots
ORDER BY nombre_lots
""")

,nombre_lots,nb_ventes,valeur_moyenne
0,1,22673,225809.0
1,2,10051,294158.0
2,3,1086,344866.0
3,4,222,446400.0
4,5,67,735296.0
5,6,34,365856.0
6,7,16,394882.0
7,8,6,1847200.0
8,9,7,928214.0
9,10,3,190717.0


Les ventes à 1 lot sont les plus nombreuses. Jusqu'à 5 lots, la valeur moyenne augmente avec le nombre de lots
(caves, parkings…) ; au-delà, chaque groupe compte moins de 40 ventes et les moyennes partent dans tous les sens :
**un effectif faible ne permet pas de conclure**.

## 3. ⚠️ Le piège WHERE ou HAVING

- `WHERE` filtre les **lignes**, avant le regroupement.
- `HAVING` filtre les **groupes**, après le calcul des agrégats.

Les types de voie qui apparaissent plus de 1 000 fois :

In [4]:
try:
    base.execute("SELECT type_voie, COUNT(*) FROM bien WHERE COUNT(*) > 1000 GROUP BY type_voie")
except sqlite3.OperationalError as erreur:
    print("Erreur de SQLite :", erreur)   # au moment du WHERE, COUNT(*) n'existe pas encore

Erreur de SQLite : misuse of aggregate: COUNT()


In [5]:
sql("""
SELECT type_voie, COUNT(*) AS nb_biens
FROM bien
GROUP BY type_voie
HAVING COUNT(*) > 1000          -- filtre sur les groupes
ORDER BY nb_biens DESC
""")

,type_voie,nb_biens
0,RUE,19580
1,AV,5107
2,BD,1842
3,ALL,1262


## 4. ⚠️ Piège : une colonne ni agrégée, ni regroupée

Beaucoup de bases refusent cette requête. **SQLite l'accepte** et affiche une valeur prise au hasard dans le groupe :

In [6]:
sql("SELECT code_departement, nom_commune, COUNT(*) AS nb_communes FROM commune GROUP BY code_departement LIMIT 3")

,code_departement,nom_commune,nb_communes
0,01,Ambérieu-en-Bugey,54
1,02,Belleu,19
2,03,Cusset,4


Pourquoi « Ambérieu-en-Bugey » pour l'Ain plutôt qu'une autre de ses communes ? Aucune raison : c'est une ligne
quelconque du groupe. Règle : toute colonne du `SELECT` est soit **dans le `GROUP BY`**, soit **dans un agrégat**.

## 5. Bonnes pratiques d'écriture

Mots-clés en **MAJUSCULES** ; une clause par ligne ; des **alias** lisibles (`AS nb_biens`) ; un commentaire `--`
pour chaque choix non évident.

### ✍️ À toi de jouer

Sur la table `bien`, donne pour chaque **nombre de pièces** des **maisons** : le nombre de maisons et la surface Carrez moyenne, en ne gardant que les nombres de pièces représentés par au moins 100 maisons.

In [7]:
# Ta réponse ici : sql(""" ... """)

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```sql
SELECT
    total_pieces,
    COUNT(*)                      AS nb_maisons,
    ROUND(AVG(surface_carrez), 1) AS surface_moyenne
FROM bien
WHERE type_local = 'Maison'
GROUP BY total_pieces
HAVING COUNT(*) >= 100
ORDER BY total_pieces;
```

Le filtre sur le type de bien porte sur les lignes (`WHERE`) ; le seuil de 100 porte sur un compte, donc sur les groupes (`HAVING`).
</details>

## ✅ À retenir

1. Ordre d'exécution : FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT.
2. `WHERE` filtre les lignes, `HAVING` filtre les groupes.
3. Toute colonne du `SELECT` est dans le `GROUP BY` ou dans un agrégat (SQLite ne te préviendra pas).
4. Majuscules, indentation, alias, commentaires : ta requête est lue par d'autres.